# Hydraulic analysis of Koto city

In this notebook you will the different parts of the hydraulic analysis of Koto city, district within Tokyo metropolis.
Data used to calculate and determine hydraulic parameters were gatehered from the following sources:
- Precipitation and other atmospheric data: JRA-3Q (Japanese Reanalysis for Three Quarters of a Century) courtesy of the United States National Science Foundation  
- Water level data: Water Information System courtesy of the Ministry of Land, Infrastructure, Transport and Tourism of Japan (Chart datum: Arakawa Peil [A.P.])
- Tidal data: Tide Table Tokyo courtesy of the Japan Meteorological Agency with an average spring tidal range of 187 cm or 1.87 m

In [21]:
#Packages used in this analysis
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats


In [22]:
#import of atmospheric data 
import xarray
filename = 'https://data.gdex.ucar.edu/d640000/kerchunk/anl_isentrop-remote-https.parq'
ds = xarray.open_dataset(filename, engine='kerchunk')
ds['bvf2-theta-an-gauss'] # Square of Brunt-Vaisala frequency

<xarray.DataArray 'bvf2-theta-an-gauss' (time: 114572, isentropic_level: 36,
                                         lat: 480, lon: 960)> Size: 8TB
[1900611993600 values with dtype=float32]
Coordinates:
  * time              (time) datetime64[ns] 917kB 1947-09-01 ... 2026-01-31T1...
  * isentropic_level  (isentropic_level) float64 288B 270.0 280.0 ... 4e+03
  * lat               (lat) float64 4kB 89.71 89.34 88.97 ... -89.34 -89.71
  * lon               (lon) float64 8kB 0.0 0.375 0.75 ... 358.9 359.2 359.6
Attributes: (12/73)
    group:                                               anl_isentrop
    jma_group_description:                               Isentropic analysis ...
    rda_group_description:                               JRA-3Q isentropic an...
    rda_group_number:                                    1
    short_name:                                          bvf2-theta-an-gauss
    jma_short_name:                                      bvf2
    ...                                                  ...
    angular_speed_of_earth_rotation:                     7.29245e-5 rad s-1
    gravitational_accleration:                           9.80665 m s-2
    gas_constant_for_dry_air:                            287.04 J K-1 kg-1
    specific_heat_of_dry_air_at_constant_pressure:       1004.6 J K-1 kg-1
    latent_heat_of_vaporization:                         2.507e+6 J kg-1
    solar_constant:                                      1365 W m-2

In [23]:
# def parse_jp_datetime(year, dt):
#     if pd.isna(dt):
#         return pd.NaT
#     dt = str(dt).replace("年", "").replace("月", "/").replace("日", "").strip()
#     if dt.endswith(" 24:00"):
#         dt = dt.replace(" 24:00", " 00:00")
#         d = pd.to_datetime(f"{year}/{dt}", format="%Y/%m/%d %H:%M", errors="coerce")
#         if pd.notna(d):
#             return d + pd.Timedelta(days=1)
#         return d
#     return pd.to_datetime(f"{year}/{dt}", format="%Y/%m/%d %H:%M", errors="coerce")

In [24]:
# # Read and process hydraulic data for Sumida River (upstream)
# from io import StringIO

# with open("iwabuchi_WL_sumida.dat", "r", encoding="cp932", errors="replace") as f:
#     lines = f.readlines()

# valid_lines = []
# for line in lines:
#     s = line.strip()
#     if not s or s.startswith("#"):
#         continue
#     if s[:4].isdigit() and "," in s:
#         valid_lines.append(s)

# data_sumida_up = "\n".join(valid_lines)

# sumida_up = pd.read_csv(
#     StringIO(data_sumida_up),
#     sep=",",
#     header=None,
#     engine="python",
#     on_bad_lines="skip",
#     na_values=["-9999.00", "$", "-", "", "0.00"])

# # Select relevant columns and rename
# sumida_up = sumida_up[[0, 1, 3, 4, 6, 8, 10, 12, 14, 15]]
# sumida_up.columns = ['year', 'maxWL', 'date_maxWL', 'wetmeanWL+STD', 'wetmeanWL',
#                       'wetmeanWL-STD', 'drymeanWL', 'minWL', 'date_minWL','meanWL']

# # Clean and convert date columns to datetime format
# sumida_up["date_maxWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(sumida_up["year"], sumida_up["date_maxWL"])]

# sumida_up["date_minWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(sumida_up["year"], sumida_up["date_minWL"])]


In [25]:
# # Read and process hydraulic data for Sumida River (downstream)
# from io import StringIO

# with open("reiganjima_WL_sumida.dat", "r", encoding="cp932", errors="replace") as f:
#     lines = f.readlines()

# valid_lines = []
# for line in lines:
#     s = line.strip()
#     if not s or s.startswith("#"):
#         continue
#     if s[:4].isdigit() and "," in s:
#         valid_lines.append(s)

# data_sumida_down = "\n".join(valid_lines)

# sumida_down = pd.read_csv(
#     StringIO(data_sumida_down),
#     sep=",",
#     header=None,
#     engine="python",
#     on_bad_lines="skip",
#     na_values=["-9999.00", "$", "-", "", "0.00"])

# # Select relevant columns and rename
# sumida_down = sumida_down[[0, 1, 3, 4, 6, 8, 10, 12, 14, 15]]
# sumida_down.columns = ['year', 'maxWL', 'date_maxWL', 'wetmeanWL+STD', 'wetmeanWL',
#                        'wetmeanWL-STD', 'drymeanWL', 'minWL', 'date_minWL','meanWL']

# # Clean and convert date columns to datetime format
# sumida_down["date_maxWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(sumida_down["year"], sumida_down["date_maxWL"])]

# sumida_down["date_minWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(sumida_down["year"], sumida_down["date_minWL"])]




In [26]:
# # Read and process hydraulic data for Arakawa River (upstream)
# from io import StringIO

# with open("nanbata_WL_arakawa.dat", "r", encoding="cp932", errors="replace") as f:
#     lines = f.readlines()

# valid_lines = []
# for line in lines:
#     s = line.strip()
#     if not s or s.startswith("#"):
#         continue
#     if s[:4].isdigit() and "," in s:
#         valid_lines.append(s)

# data_arakawa_up = "\n".join(valid_lines)

# arakawa_up = pd.read_csv(
#     StringIO(data_arakawa_up),
#     sep=",",
#     header=None,
#     engine="python",
#     on_bad_lines="skip",
#     na_values=["-9999.00", "$", "-", "", "0.00"])

# # Select relevant columns and rename
# arakawa_up = arakawa_up[[0, 1, 3, 4, 6, 8, 10, 12, 14, 15]]
# arakawa_up.columns = ['year', 'maxWL', 'date_maxWL', 'wetmeanWL+STD', 'wetmeanWL',
#                       'wetmeanWL-STD', 'drymeanWL', 'minWL', 'date_minWL','meanWL']

# # Clean and convert date columns to datetime format
# arakawa_up["date_maxWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(arakawa_up["year"], arakawa_up["date_maxWL"])]

# arakawa_up["date_minWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(arakawa_up["year"], arakawa_up["date_minWL"])]

In [27]:
# # Read and process hydraulic data for Arakawa River (midstream)
# from io import StringIO

# with open("nishiarai_WL_arakawa.dat", "r", encoding="cp932", errors="replace") as f:
#     lines = f.readlines()

# valid_lines = []
# for line in lines:
#     s = line.strip()
#     if not s or s.startswith("#"):
#         continue
#     if s[:4].isdigit() and "," in s:
#         valid_lines.append(s)

# data_arakawa_mid = "\n".join(valid_lines)

# arakawa_mid = pd.read_csv(
#     StringIO(data_arakawa_mid),
#     sep=",",
#     header=None,
#     engine="python",
#     on_bad_lines="skip",
#     na_values=["-9999.00", "$", "-", "", "0.00"])


# # Select relevant columns and rename
# arakawa_mid = arakawa_mid[[0, 1, 3, 4, 6, 8, 10, 12, 14, 15]]
# arakawa_mid.columns = ['year', 'maxWL', 'date_maxWL', 'wetmeanWL+STD', 'wetmeanWL',
#                        'wetmeanWL-STD', 'drymeanWL', 'minWL', 'date_minWL','meanWL']

# # Clean and convert date columns to datetime format
# arakawa_mid["date_maxWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(arakawa_mid["year"], arakawa_mid["date_maxWL"])]

# arakawa_mid["date_minWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(arakawa_mid["year"], arakawa_mid["date_minWL"])]


In [28]:
# # Read and process hydraulic data for Arakawa River (downstream)
# from io import StringIO

# with open("onagigawa_WL_arakawa.dat", "r", encoding="cp932", errors="replace") as f:
#     lines = f.readlines()

# valid_lines = []
# for line in lines:
#     s = line.strip()
#     if not s or s.startswith("#"):
#         continue
#     if s[:4].isdigit() and "," in s:
#         valid_lines.append(s)

# data_arakawa_down = "\n".join(valid_lines)

# arakawa_down = pd.read_csv(
#     StringIO(data_arakawa_down),
#     sep=",",
#     header=None,
#     engine="python",
#     on_bad_lines="skip",
#     na_values=["-9999.00", "$", "-", "", "0.00"])

# # Select relevant columns and rename
# arakawa_down = arakawa_down[[0, 1, 3, 4, 6, 8, 10, 12, 14, 15]]
# arakawa_down.columns = ['year', 'maxWL', 'date_maxWL', 'wetmeanWL+STD', 'wetmeanWL',
#                         'wetmeanWL-STD', 'drymeanWL', 'minWL', 'date_minWL','meanWL']

# # Clean and convert date columns to datetime format
# arakawa_down["date_maxWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(arakawa_down["year"], arakawa_down["date_maxWL"])]

# arakawa_down["date_minWL"] = [parse_jp_datetime(y, d)
#     for y, d in zip(arakawa_down["year"], arakawa_down["date_minWL"])]


In [29]:
# #Saving dataframes to CSV files
# sumida_up.to_csv('sumida_upstream_iwabuchi.csv', index=False)
# sumida_down.to_csv('sumida_downstream_reiganjima.csv', index=False)
# arakawa_up.to_csv('arakawa_upstream_nanbata.csv', index=False)
# arakawa_mid.to_csv('arakawa_midstream_nishiarai.csv', index=False)
# arakawa_down.to_csv('arakawa_downstream_onagigawa.csv', index=False)

In [30]:
#Load csv files into dataframes
sumida_up = pd.read_csv('sumida_upstream_iwabuchi.csv')
sumida_down = pd.read_csv('sumida_downstream_reiganjima.csv')
arakawa_up = pd.read_csv('arakawa_upstream_nanbata.csv')
arakawa_mid = pd.read_csv('arakawa_midstream_nishiarai.csv')
arakawa_down = pd.read_csv('arakawa_downstream_onagigawa.csv')